## Week 6 Day 3 へようこそ: MCP によるコンテキストエンジニアリング

かつては Prompt Engineering(プロンプトエンジニアリング)が話題でしたが、今ではそれに代わって新しいスキル「Context Engineering(コンテキストエンジニアリング)」が注目されています。

Google DeepMind の Philipp Schmid は、コンテキストエンジニアリングに関する画期的な記事を書いています。

https://www.philschmid.de/context-engineering

今週は、MCP サーバーを大いに活用しながら、コンテキストエンジニアリングを実践していきます。

1. 長期記憶: エージェントが書き込み、後で読み返す知識グラフ
2. Web 検索: 生きた Web からの最新情報
3. エージェント型 RAG: エージェントが自分の調査で埋めていき、検索するベクトルストア
4. 統合: ライブの外部サービスへの接続、ローカルなフォールバック付き

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio, create_static_tool_filter
import os
from pathlib import Path
from datetime import datetime
from IPython.display import Markdown, display

load_dotenv(override=True)

In [ ]:
# メインモデルをOpenAIからGeminiに切り替える
from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel, set_tracing_disabled

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini_client = AsyncOpenAI(api_key=os.getenv("GOOGLE_API_KEY"), base_url=GEMINI_BASE_URL)
MODEL_NAME = OpenAIChatCompletionsModel(model="gemini-flash-latest", openai_client=gemini_client)

# GeminiのキーではOpenAIのトレース機能(platform.openai.com/traces)は使えないため無効化
set_tracing_disabled(True)


### Windows での注意点

ノートブックからローカルの MCP サーバーを起動すると、Windows では厄介な問題にぶつかります。サーバーは起動時の出力を stderr に書き込みますが、Windows の Jupyter カーネルの中ではこのストリームの裏に実際のファイルハンドルがないため、起動が `io.UnsupportedOperation: fileno` エラーで失敗します。一方、Mac と Linux では影響がありません。

対処法は、サーバーの stderr をヌルデバイスに送ることです。これにより、サーバーは常に書き込める実在の場所を持つことになります。次のセルでこれを一度だけ行えば、以降のすべてのセルで、OpenAI Agents SDK のドキュメントどおりに `MCPServerStdio` をそのまま使えるようになります。Mac と Linux では、サーバーの起動時バナーがノートブックに表示されなくなるだけで、他に影響はありません。

In [ ]:
# Windows では、Jupyter カーネルから起動した stdio MCP サーバーが、実際のファイルディスクリプタを
# 持たない stderr ストリームに書き込もうとして io.UnsupportedOperation: fileno でクラッシュする。
# サーバーの stderr をヌルデバイスに送ることで、常に書き込める実在の場所を用意し、これにより
# 以降のすべてのセルで OpenAI Agents SDK のドキュメントどおりに MCPServerStdio を使えるようにする。
# Mac と Linux では影響がない。
import functools
import subprocess
import agents.mcp.server

agents.mcp.server.stdio_client = functools.partial(agents.mcp.server.stdio_client, errlog=subprocess.DEVNULL)

## パート1: 長期記憶

最初のコンテキストソースは記憶です。これは公式の知識グラフサーバーで、エンティティ、そのエンティティに関する観測事項、そしてそれらの間の関係を保存し、実行の間もディスク上に保持します。ここでは `memory/memory.json` を指定するので、そのファイルを開けば、構築されたグラフを読むことができます。

これにより、エージェントは普段は持っていないもの、つまり会話よりも長く生き続ける記憶を手に入れます。エージェントは事実を学ぶたびに書き込み、後でそれを読み返します。

https://github.com/modelcontextprotocol/servers/tree/main/src/memory

In [ ]:
memory_path = os.path.abspath("memory/memory.json")
memory_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-memory"], "env": {"MEMORY_FILE_PATH": memory_path}}

async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=60) as server:
    memory_tools = await server.list_tools()

memory_tools

In [ ]:
instructions = "You use your entity tools as a persistent memory to store and recall information about your conversations."
request = "My name's Ed. I'm an LLM engineer. I'm teaching a course about AI Agents, including the incredible MCP protocol. MCP is a protocol for connecting agents with tools, resources and prompt templates, and makes it easy to integrate AI agents with capabilities."
model = MODEL_NAME

In [ ]:
async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

In [ ]:
async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, "My name's Ed. What do you know about me?")
    display(Markdown(result.final_output))

### トレースを確認してみましょう

https://platform.openai.com/traces

## パート2: Web 検索

モデルは、学習に使われたデータしか知りません。Web 検索は、モデルを最新の状態に保つためのコンテキストソースです。

ここでは、エージェント向けに作られた検索 API である Tavily を使います。これは、LLM がそのまま使える、整理されランク付けされた綺麗な結果を返してくれます。Tavily は自前の MCP サーバーを保守しているので、接続はほんの一行で済みます。

これには無料の API キーが必要です。

1. https://www.tavily.com でサインアップします
2. 無料プランでは、クレジットカード不要で月1,000回の検索が可能です
3. API キー(`tvly-` で始まります)をコピーして、`.env` ファイルに追加します。

`TAVILY_API_KEY=tvly-xxxx`

In [ ]:
tavily_params = {"command": "npx", "args": ["-y", "tavily-mcp@latest"], "env": {"TAVILY_API_KEY": os.getenv("TAVILY_API_KEY")}}

async with MCPServerStdio(params=tavily_params, client_session_timeout_seconds=60) as server:
    tavily_tools = await server.list_tools()

tavily_tools

Tavily のサーバーは、複数のツール(search、extract、crawl、map、research)を提供しています。このラボでは普通の Web 検索だけが必要なので、サーバーを `tavily_search` に制限します。OpenAI Agents SDK では、静的なツールフィルターを使って、選んだツールだけをエージェントに渡すことができます。このようにエージェントのツールを厳選することも、それ自体がコンテキストエンジニアリングです。

In [ ]:
instructions = "You search the web for information and briefly summarize the takeaways."
request = f"Please research the latest news on Amazon stock price and briefly summarize its outlook. For context, the current date is {datetime.now().strftime('%Y-%m-%d')}"
model = MODEL_NAME
search_only = create_static_tool_filter(allowed_tool_names=["tavily_search"])

In [ ]:
async with MCPServerStdio(params=tavily_params, client_session_timeout_seconds=60, tool_filter=search_only) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

## パート3: エージェント型 RAG

RAG(検索拡張生成)とは、モデルに関連する文書を与えて、その回答を裏付けさせることです。通常のセットアップでは、事前に文書をベクトルストアに読み込んでおきます。エージェント型 RAG はこれを逆転させます。エージェント自身が知識ベースを構築し、何を保存する価値があるかを判断しながら、作業の中でそれを蓄積していくのです。

ここでは、公式の Qdrant MCP サーバーを使います。Qdrant はベクトルデータベースで、このサーバーは2つのツールを公開しています。1つはテキストの断片を保存し、もう1つは、あるクエリに対して保存済みのテキストの中から最も関連性の高いものを見つけます。ここでは完全にローカルで動作します。`QDRANT_LOCAL_PATH` によってすべてがディスク上に保持され、別途データベースを動かす必要はありません。また、テキストの埋め込みにはローカルモデルを使うので、追加の API キーも不要です。最初の保存または検索のときにその小さな埋め込みモデルをダウンロードするので、初回利用時だけ一度停止します。

https://github.com/qdrant/mcp-server-qdrant

Tavily と Qdrant の両方を1つのエージェントに与えます。エージェントは Web 上でトピックを調査し、学んだことを保存し、その後は自分自身の知識ベースから答えます。

In [ ]:
vectordb_path = Path("memory/qdrant")
vectorstore_params = {
    "command": "uvx",
    "args": ["mcp-server-qdrant"],
    "env": {
        "QDRANT_LOCAL_PATH": str(vectordb_path),
        "COLLECTION_NAME": "knowledge",
    },
}

async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as server:
    vectorstore_tools = await server.list_tools()

vectorstore_tools

In [ ]:
INSTRUCTIONS = """You research topics on the web and build up a knowledge base for later.
When you learn something worth keeping, store it in your knowledge base.
When you are asked what you know, search your knowledge base and answer from it."""

model = MODEL_NAME

async with MCPServerStdio(params=tavily_params, client_session_timeout_seconds=60, tool_filter=search_only) as search_server:
    async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as vector_server:
        agent = Agent(name="researcher", instructions=INSTRUCTIONS, model=model, mcp_servers=[search_server, vector_server])
        with trace("research and store"):
            result = await Runner.run(agent, "Research the latest news on Nvidia and store the key facts in your knowledge base.", max_turns=20)
        display(Markdown(result.final_output))

下のエージェントは知識ベースしか持たず、Web 検索は持っていません。Nvidia について何を答えるにしても、それは最初のエージェントが保存した内容を思い出しているだけです。それが RAG の「検索」の側面です。

In [ ]:
async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as vector_server:
    agent = Agent(name="researcher", instructions=INSTRUCTIONS, model=model, mcp_servers=[vector_server])
    with trace("retrieve"):
        result = await Runner.run(agent, "Based on your knowledge base, what's the latest on Nvidia?")
    display(Markdown(result.final_output))

### トレースを確認してみましょう

https://platform.openai.com/traces

## パート4: 統合

最後のコンテキストソースは、ライブの外部サービスです。ここでは、有名な金融データプロバイダーであり、自前の MCP サーバーを公開している Massive(旧 Polygon.io)からの市場データを使います。

このセットアップは任意です。Massive はクレジットカード不要の無料 API キーを提供しており、それを使うと本物の終値の市場データが得られます。

1. https://www.massive.com でサインアップします
2. API キーを作成します
3. `.env` ファイルに追加します。

`MASSIVE_API_KEY=xxxx`

サインアップしたくない場合は、次のセルは Day 2 で作ったのと同じ種類の MCP サーバーであるローカルの市場サーバーにフォールバックし、シミュレートされた価格を提供するので、ラボの残りの部分もそのまま動作します。

In [ ]:
massive_api_key = os.getenv("MASSIVE_API_KEY")

if massive_api_key:
    market_params = {
        "command": "uvx",
        # mcp_massive は今でも mcp.server.fastmcp をインポートしているが、これは mcp SDK の 2.0.0 リリースで
        # 削除された API である。mcp_massive は mcp の依存バージョンを上限指定していないので、
        # ここでは uvx を古い、互換性のある mcp に固定する。
        "args": ["--with", "mcp<2.0.0", "--from", "git+https://github.com/massive-com/mcp_massive@v0.10.0", "mcp_massive"],
        "env": {"MASSIVE_API_KEY": massive_api_key},
    }
else:
    market_params = {"command": "uv", "args": ["run", "-m", "backend.market_server"]}

async with MCPServerStdio(params=market_params, client_session_timeout_seconds=120) as server:
    market_tools = await server.list_tools()

market_tools

In [ ]:
instructions = "You answer questions about the stock market."
request = "What was the most recent price that Apple (AAPL) traded at?"
model = MODEL_NAME

async with MCPServerStdio(params=market_params, client_session_timeout_seconds=120) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

## 今日はここまでです

4つの MCP サーバー、4種類のコンテキスト。長期記憶、Web 検索、エージェントが自分で構築する知識ベース、そしてライブの統合です。エージェントにどのソースが必要かを選び、それらを組み込むことこそ、実践としてのコンテキストエンジニアリングの大部分を占めます。

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">エクササイズ</h2>
            <span style="color:#ff7800;">glama.ai/mcp や smithery.ai のような MCP マーケットプレイスを眺めてみて、今日の4つのパターンのうちどれかを使って、このノートブックに別のコンテキストソースを追加してみましょう。
            </span>
        </td>
    </tr>
</table>